In [0]:
from pyspark.sql.functions import col, count, avg, round

# Load final inference results
df = spark.table("gold_db.gold_inference_results")

print("Inference results loaded:", df.count(), "banks")
display(df.limit(3))

Inference results loaded: 108 banks


CERT,risk_tier,failed,failure_probability,contagion_risk,avg_asset,avg_capital_ratio,avg_leverage_ratio,pagerank,betweenness_centrality,inference_timestamp,model_version
10004,LOW,0,0.012256285240747502,LOW,81541.51785714286,0.11874583333333338,8.72588452380952,0.008770303993570245,0.12454631623531502,2026-03-13T17:20:05.259Z,random_forest_v1
10013,MEDIUM,0,0.001727142076632539,LOW,157259.5294117647,0.08050196078431372,12.612827450980394,0.007845867545949734,0.09725910709938193,2026-03-13T17:20:05.259Z,random_forest_v1
10119,MEDIUM,0,0.050999838199416746,LOW,17253.880597014926,0.09310149253731342,12.027438805970151,0.014436710277083242,0.021608154747142432,2026-03-13T17:20:05.259Z,random_forest_v1


In [0]:
# Risk distribution across all banks
print("Risk Distribution:")
display(
    df.groupBy("contagion_risk")
    .count()
    .orderBy("count", ascending=False)
)

Risk Distribution:


contagion_risk,count
LOW,100
CRITICAL,7
HIGH,1


In [0]:
# Top 10 most dangerous banks
print("Top 10 Most Dangerous Banks:")
display(
    df.select("CERT", "contagion_risk", "failure_probability", 
              "avg_asset", "avg_capital_ratio", "avg_leverage_ratio")
    .orderBy("failure_probability", ascending=False)
    .limit(10)
)

Top 10 Most Dangerous Banks:


CERT,contagion_risk,failure_probability,avg_asset,avg_capital_ratio,avg_leverage_ratio
10085,CRITICAL,0.9977714285714286,14460.333333333334,0.04805,11.097850000000001
10086,CRITICAL,0.978586975343497,133159.73394495412,0.07500366972477059,14.185927522935774
10116,CRITICAL,0.9770608891108892,14418.857142857143,0.08270000000000001,12.363457142857142
10054,CRITICAL,0.9763073417987472,170200.3173076923,0.1001846153846154,10.63728269230769
10094,CRITICAL,0.9477714285714286,37756.46153846154,0.057061538461538455,-8.133038461538462
10155,CRITICAL,0.9212106497523666,31369.3125,0.08999375,7.6420812499999995
1006,CRITICAL,0.8758502603461656,102857.92233009709,0.09231941747572818,11.923082524271841
10100,HIGH,0.7280670995670997,1976400.3039215687,0.0836980392156862,12.649467647058822
10153,LOW,0.1674306478405316,959962.6666666666,0.06199629629629629,16.222255555555552
10093,LOW,0.1397266004683172,162915.76,0.06967733333333335,20.89233466666666


In [0]:
# Average failure probability by risk tier
print("Average Failure Probability by Risk Tier:")
display(
    df.groupBy("risk_tier")
    .agg(
        count("CERT").alias("total_banks"),
        avg("failure_probability").alias("avg_failure_prob"),
        avg("avg_asset").alias("avg_assets")
    )
    .orderBy("avg_failure_prob", ascending=False)
)

Average Failure Probability by Risk Tier:


risk_tier,total_banks,avg_failure_prob,avg_assets
CRITICAL,2,0.5687490145198729,88688.04666666668
MEDIUM,51,0.11111369149669244,169082.57421382744
HIGH,25,0.10882104599234188,206175.24116530005
LOW,30,0.01662123197007136,239383.54364811568


In [0]:
# Register table for SQL queries
df.createOrReplaceTempView("bank_risk_view")

# Banks that are low risk tier but high failure probability
print("Hidden Risk Banks — LOW tier but HIGH failure probability:")
display(spark.sql("""
    SELECT CERT, risk_tier, contagion_risk,
           failure_probability, avg_asset, avg_capital_ratio
    FROM bank_risk_view
    WHERE risk_tier = 'LOW' AND failure_probability > 0.3
    ORDER BY failure_probability DESC
"""))

Hidden Risk Banks — LOW tier but HIGH failure probability:


CERT,risk_tier,contagion_risk,failure_probability,avg_asset,avg_capital_ratio


In [0]:
# Most connected dangerous banks
print("Most Connected Dangerous Banks:")
display(spark.sql("""
    SELECT CERT, contagion_risk, failure_probability,
           pagerank, betweenness_centrality, avg_asset
    FROM bank_risk_view
    WHERE contagion_risk IN ('CRITICAL', 'HIGH')
    ORDER BY pagerank DESC
"""))

Most Connected Dangerous Banks:


CERT,contagion_risk,failure_probability,pagerank,betweenness_centrality,avg_asset
10100,HIGH,0.7280670995670997,0.01375493102796586,8.751638489141471E-5,1976400.3039215687
10085,CRITICAL,0.9977714285714286,0.011563152088114923,0.018703749870788464,14460.333333333334
10094,CRITICAL,0.9477714285714286,0.011041446628460665,0.08698216576775972,37756.46153846154
10155,CRITICAL,0.9212106497523666,0.009372319221798845,0.07608235834723899,31369.3125
10116,CRITICAL,0.9770608891108892,0.00918296727688933,0.018146077224631758,14418.857142857143
10086,CRITICAL,0.978586975343497,0.008661905076631891,0.10545859110185583,133159.73394495412
10054,CRITICAL,0.9763073417987472,0.007302859347841807,0.08698216576775956,170200.3173076923
1006,CRITICAL,0.8758502603461656,0.007124272804838961,0.11455665452116778,102857.92233009709


In [0]:
# Save final analytics summary
df_analytics = df.groupBy("risk_tier", "contagion_risk") \
    .agg(
        count("CERT").alias("total_banks"),
        avg("failure_probability").alias("avg_failure_prob"),
        avg("avg_asset").alias("avg_assets"),
        avg("avg_capital_ratio").alias("avg_capital_ratio"),
        avg("pagerank").alias("avg_pagerank")
    )

df_analytics.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("gold_db.gold_analytics_summary")

print("gold_analytics_summary saved:", spark.table("gold_db.gold_analytics_summary").count(), "rows")

gold_analytics_summary saved: 8 rows
